# 01b · EverydayScents: R1 raw features

**Purpose.** Downsample each sensor curve to T' = 70 and flatten (R1). Fit the 4 heads on both tasks with StratifiedGroupKFold(5), groups = session, 3 seeds.

**Inputs**
- EverydayScents `X_signals.npy` (144, 11, 210) + `meta.csv`

**Outputs**
- F_raw: (144, 11, 70) → (144, 770)
- `results/01b_es_raw.csv`: one row per seed, fold, task, rep, head, metrics

In [1]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import TASKS, load_baseline_level, load_processed, precomputed, run_cv, save_results, summarize
from src.features.raw import raw_features
DATASET = "es"
REP = "R1_raw"
NAME = "01b_es_raw"          # results/01b_es_raw.csv

X, meta, info = load_processed(DATASET)    # X: (N, C, T) baseline corrected
print(X.shape, meta["y_multi"].value_counts().sort_index().tolist())

(144, 11, 210) [33, 39, 50, 22]


## R1: downsample every curve to T' = 70 points and flatten
Nothing is fitted, so the features are computed once for all samples.

In [2]:
T_PRIME = 70
F = raw_features(X, T_PRIME)               # (N, C * T') = (144, 770)
assert F.shape == (144, 770)
F.shape

(144, 770)

## Cross validation: 4 heads × 5 folds × 3 seeds, both tasks
Everything that is fitted (scaler, heads, hyperparameters) sees the training fold only.

In [3]:
scores, preds = [], []
for task in TASKS:                                   # "multi", then "binary"
    s, p = run_cv(F, meta, DATASET, task, REP, precomputed)
    scores.append(s)
    preds.append(p)
scores, preds = pd.concat(scores, ignore_index=True), pd.concat(preds, ignore_index=True)
save_results(scores, preds, NAME)
print("saved results/" + NAME + ".csv")

es multi R1_raw seed 0 fold 0: logreg 0.824  svm 0.816  rf 0.821  mlp 0.790
es multi R1_raw seed 0 fold 1: logreg 0.796  svm 0.884  rf 0.671  mlp 0.753
es multi R1_raw seed 0 fold 2: logreg 0.843  svm 0.916  rf 0.914  mlp 0.862
es multi R1_raw seed 0 fold 3: logreg 0.608  svm 0.608  rf 0.747  mlp 0.612
es multi R1_raw seed 0 fold 4: logreg 0.853  svm 0.856  rf 0.861  mlp 0.775
es multi R1_raw seed 1 fold 0: logreg 0.768  svm 0.729  rf 0.791  mlp 0.703
es multi R1_raw seed 1 fold 1: logreg 0.813  svm 0.854  rf 0.676  mlp 0.877
es multi R1_raw seed 1 fold 2: logreg 0.802  svm 0.852  rf 0.942  mlp 0.694
es multi R1_raw seed 1 fold 3: logreg 0.716  svm 0.685  rf 0.729  mlp 0.683
es multi R1_raw seed 1 fold 4: logreg 0.872  svm 0.853  rf 0.916  mlp 0.841
es multi R1_raw seed 2 fold 0: logreg 0.965  svm 0.867  rf 0.794  mlp 0.923
es multi R1_raw seed 2 fold 1: logreg 0.771  svm 0.569  rf 0.844  mlp 0.665
es multi R1_raw seed 2 fold 2: logreg 0.735  svm 0.796  rf 0.851  mlp 0.586
es multi R1_

## Results: mean ± std over 3 seeds × 5 folds

In [4]:
print("macro F1"); display(summarize(scores))
print("balanced accuracy"); display(summarize(scores, "balanced_acc"))

macro F1


head                          logreg            mlp             rf  \
dataset task   rep                                                   
es      binary R1_raw  0.928 ± 0.050  0.946 ± 0.050  0.953 ± 0.058   
        multi  R1_raw  0.788 ± 0.087  0.738 ± 0.104  0.817 ± 0.088   

head                             svm  
dataset task   rep                    
es      binary R1_raw  0.885 ± 0.073  
        multi  R1_raw  0.789 ± 0.111

balanced accuracy


head                          logreg            mlp             rf  \
dataset task   rep                                                   
es      binary R1_raw  0.927 ± 0.051  0.947 ± 0.050  0.954 ± 0.061   
        multi  R1_raw  0.807 ± 0.072  0.755 ± 0.092  0.835 ± 0.077   

head                             svm  
dataset task   rep                    
es      binary R1_raw  0.878 ± 0.081  
        multi  R1_raw  0.793 ± 0.089

In [5]:
print("ROC AUC, binary task"); summarize(scores[scores["task"] == "binary"], "roc_auc")

ROC AUC, binary task


,,head,logreg,mlp,rf,svm
dataset,task,rep,,,,
es,binary,R1_raw,0.973 ± 0.030,0.970 ± 0.040,0.981 ± 0.034,0.972 ± 0.041
